In [1]:
import pandas as pd
from google.colab import drive
drive.mount('/content/drive')

BASE_PATH = '/content/drive/MyDrive/Cadetx Project/HeavySuppliersWarehouseDatasets/'

ledger = pd.read_csv(BASE_PATH + 'stock_ledger.csv')
inv    = pd.read_csv(BASE_PATH + 'inventory_master.csv')
po_h   = pd.read_csv(BASE_PATH + 'purchase_orders_header.csv')
po_l   = pd.read_csv(BASE_PATH + 'purchase_orders_lines.csv')
so_h   = pd.read_csv(BASE_PATH + 'sales_orders_header.csv')
so_l   = pd.read_csv(BASE_PATH + 'sales_orders_lines.csv')

print('ledger', len(ledger), '| inv', len(inv), '| po_h', len(po_h), '| po_l', len(po_l), '| so_h', len(so_h), '| so_l', len(so_l))

Mounted at /content/drive
ledger 237230 | inv 180 | po_h 24000 | po_l 155495 | so_h 20000 | so_l 130402


In [2]:
sales = so_l.merge(so_h[['so_id', 'branch_id', 'order_status', 'delivery_date']], on='so_id')
sales = sales[sales['order_status'] == 'Delivered'].copy()
sales['delivery_date'] = pd.to_datetime(sales['delivery_date'])

last_sale = sales.groupby(['product_id', 'branch_id'])['delivery_date'].max()
end_date = sales['delivery_date'].max()
days_since = (end_date - last_sale).dt.days

print('Last delivery date in the data:', end_date.date())
print('Pairs with a Delivered sale:', len(last_sale), 'of 180')
print('Pairs with no Delivered sale:', 180 - len(last_sale))
print()
print(days_since.describe())
print()
for n in [7, 14, 30, 60, 90, 180]:
    print('Pairs with no sale in the last', n, 'days:', (days_since > n).sum())
print()
print('5 pairs furthest behind:')
print(days_since.sort_values(ascending=False).head(5))

Last delivery date in the data: 2025-01-13
Pairs with a Delivered sale: 180 of 180
Pairs with no Delivered sale: 0

count    180.000000
mean       6.816667
std        5.454202
min        0.000000
25%        2.750000
50%        6.000000
75%       11.000000
max       27.000000
Name: delivery_date, dtype: float64

Pairs with no sale in the last 7 days: 62
Pairs with no sale in the last 14 days: 13
Pairs with no sale in the last 30 days: 0
Pairs with no sale in the last 60 days: 0
Pairs with no sale in the last 90 days: 0
Pairs with no sale in the last 180 days: 0

5 pairs furthest behind:
product_id  branch_id
P023        DEL001       27
P011        HYD001       26
P012        HYD001       25
P017        PUN001       23
P012        DEL001       22
Name: delivery_date, dtype: int64


In [3]:
one_year_ago = end_date - pd.Timedelta(days=365)
recent = sales[sales['delivery_date'] > one_year_ago]

last_year = recent.groupby(['product_id', 'branch_id']).agg(
    units_sold=('quantity', 'sum'),
    order_lines=('so_id', 'count')
)

print('Window:', one_year_ago.date(), 'to', end_date.date())
print('Pairs with sales in the window:', len(last_year), 'of 180')
print()
print(last_year['units_sold'].describe())
print()
print('5 pairs that sold the fewest units:')
print(last_year.sort_values('units_sold').head(5))

Window: 2024-01-14 to 2025-01-13
Pairs with sales in the window: 180 of 180

count     180.000000
mean     1096.111111
std       183.109208
min       703.000000
25%       950.750000
50%      1082.000000
75%      1230.000000
max      1548.000000
Name: units_sold, dtype: float64

5 pairs that sold the fewest units:
                      units_sold  order_lines
product_id branch_id                         
P015       PUN001            703           67
P005       HYD001            715           72
P004       HYD001            729           67
P019       HYD001            743           71
P008       HYD001            793           79


In [4]:
cover = last_year.join(inv.set_index(['product_id', 'branch_id'])['current_stock'])
cover['years_of_cover'] = cover['current_stock'] / cover['units_sold']

print('Pairs:', len(cover))
print('Pairs with missing current_stock:', cover['current_stock'].isna().sum())
print()
print(cover['years_of_cover'].describe())
print()
print('5 pairs with the highest years of cover:')
print(cover.sort_values('years_of_cover', ascending=False).head(5))
print()
print('5 pairs with the lowest years of cover:')
print(cover.sort_values('years_of_cover').head(5))

Pairs: 180
Pairs with missing current_stock: 0

count    180.000000
mean     100.681189
std       18.193311
min       64.754131
25%       86.515677
50%       98.360732
75%      114.181724
max      157.853147
Name: years_of_cover, dtype: float64

5 pairs with the highest years of cover:
                      units_sold  order_lines  current_stock  years_of_cover
product_id branch_id                                                        
P005       HYD001            715           72         112865      157.853147
P015       PUN001            703           67         106048      150.850640
P004       HYD001            729           67         109943      150.813443
P018       HYD001            809           82         112953      139.620519
P002       HYD001            880           85         121013      137.514773

5 pairs with the lowest years of cover:
                      units_sold  order_lines  current_stock  years_of_cover
product_id branch_id                                    

In [5]:
by_branch = cover.groupby('branch_id')['years_of_cover'].agg(['count', 'mean', 'min', 'max']).round(1)
print(by_branch.sort_values('mean', ascending=False))

           count   mean   min    max
branch_id                           
HYD001        30  119.9  89.2  157.9
PUN001        30  108.7  84.1  150.9
DEL001        30  105.5  80.9  129.5
AHM001        30  101.7  79.2  126.8
KOL001        30   86.4  72.4  111.4
CHN001        30   81.8  64.8  101.3


In [6]:
for n in [1, 2, 5, 10, 50, 100]:
    print('Pairs with more than', n, 'years of cover:', (cover['years_of_cover'] > n).sum(), 'of 180')

Pairs with more than 1 years of cover: 180 of 180
Pairs with more than 2 years of cover: 180 of 180
Pairs with more than 5 years of cover: 180 of 180
Pairs with more than 10 years of cover: 180 of 180
Pairs with more than 50 years of cover: 180 of 180
Pairs with more than 100 years of cover: 84 of 180


In [7]:
by_branch2 = cover.groupby('branch_id')[['current_stock', 'units_sold']].mean().round(0)
by_branch2['years_of_cover'] = (by_branch2['current_stock'] / by_branch2['units_sold']).round(1)
print(by_branch2.sort_values('years_of_cover', ascending=False))

           current_stock  units_sold  years_of_cover
branch_id                                           
HYD001          108176.0       916.0           118.1
PUN001          108542.0      1013.0           107.1
DEL001          109622.0      1052.0           104.2
AHM001          105166.0      1046.0           100.5
KOL001          104127.0      1218.0            85.5
CHN001          107808.0      1331.0            81.0


In [8]:
# 1. Ledger chain: start each pair from opening_stock and measure each row's balance change
led = ledger.copy()
led['movement_date'] = pd.to_datetime(led['movement_date'])
led['_order'] = range(len(led))
led = led.merge(inv[['product_id', 'branch_id', 'opening_stock']], on=['product_id', 'branch_id'], how='left')
led = led.sort_values(['product_id', 'branch_id', 'movement_date', '_order'])
led['prev'] = led.groupby(['product_id', 'branch_id'])['running_balance'].shift(1)
led['prev'] = led['prev'].fillna(led['opening_stock'])
led['change'] = led['running_balance'] - led['prev']

is_in = led['movement_type'] == 'IN'
is_out = led['movement_type'] == 'OUT'
is_adj = led['movement_type'] == 'ADJUSTMENT'
print('IN rows:', is_in.sum(), '| change = +quantity:', (led.loc[is_in, 'change'] == led.loc[is_in, 'quantity']).sum())
print('OUT rows:', is_out.sum(), '| change = -quantity:', (led.loc[is_out, 'change'] == -led.loc[is_out, 'quantity']).sum())
print('ADJUSTMENT rows:', is_adj.sum(), '| change = +/- quantity:', (led.loc[is_adj, 'change'].abs() == led.loc[is_adj, 'quantity']).sum())

# 2. Net adjustment per pair
net_adj = led[is_adj].groupby(['product_id', 'branch_id'])['change'].sum()
print('Net adjustments, all pairs:', net_adj.sum())

# 3. Order-line stock per pair: opening + received - delivered + net adjustments
received = po_l.merge(po_h[['po_id', 'branch_id', 'po_status']], on='po_id')
received = received[received['po_status'] == 'Received'].groupby(['product_id', 'branch_id'])['quantity'].sum()
delivered = sales.groupby(['product_id', 'branch_id'])['quantity'].sum()

stock = inv.set_index(['product_id', 'branch_id'])[['opening_stock', 'current_stock']].copy()
stock['received'] = received
stock['delivered'] = delivered
stock['net_adj'] = net_adj
stock = stock.fillna(0)
stock['order_line_stock'] = stock['opening_stock'] + stock['received'] - stock['delivered'] + stock['net_adj']

print('Pairs:', len(stock))
print('Order-line stock, all pairs:', int(stock['order_line_stock'].sum()))

IN rows: 127611 | change = +quantity: 127611
OUT rows: 107165 | change = -quantity: 107165
ADJUSTMENT rows: 2454 | change = +/- quantity: 2454
Net adjustments, all pairs: 62.0
Pairs: 180
Order-line stock, all pairs: 21185215


In [9]:
c2 = last_year[['units_sold']].join(stock[['order_line_stock']])
c2['years_of_cover'] = c2['order_line_stock'] / c2['units_sold']

print('Pairs:', len(c2))
print('Empty values:', c2.isna().sum().sum(), '| Negative stock:', (c2['order_line_stock'] < 0).sum())
print()
print(c2['years_of_cover'].describe())
print()
print('5 pairs with the highest years of cover:')
print(c2.sort_values('years_of_cover', ascending=False).head(5))
print()
print('5 pairs with the lowest years of cover:')
print(c2.sort_values('years_of_cover').head(5))
print()
for n in [100, 120, 150]:
    print('Pairs with more than', n, 'years of cover:', (c2['years_of_cover'] > n).sum(), 'of 180')

Pairs: 180
Empty values: 0 | Negative stock: 0

count    180.000000
mean     110.461331
std       19.633153
min       72.035030
25%       96.293474
50%      108.235778
75%      124.366537
max      171.921678
Name: years_of_cover, dtype: float64

5 pairs with the highest years of cover:
                      units_sold  order_line_stock  years_of_cover
product_id branch_id                                              
P005       HYD001            715          122924.0      171.921678
P004       HYD001            729          120019.0      164.635117
P015       PUN001            703          115509.0      164.308677
P018       HYD001            809          123601.0      152.782447
P002       HYD001            880          132967.0      151.098864

5 pairs with the lowest years of cover:
                      units_sold  order_line_stock  years_of_cover
product_id branch_id                                              
P027       CHN001           1513          108989.0       72.035030
P0

In [10]:
def band(y):
    if y > 150:
        return '1 Highest (above 150 years)'
    elif y > 120:
        return '2 High (120 to 150 years)'
    elif y > 100:
        return '3 Medium (100 to 120 years)'
    else:
        return '4 Lower (up to 100 years)'

c2['band'] = c2['years_of_cover'].apply(band)

print(c2['band'].value_counts().sort_index())
print('Total:', c2['band'].value_counts().sum())
print()
print(pd.crosstab(c2.index.get_level_values('branch_id'), c2['band']))
print()
print('The pairs in the highest band:')
print(c2[c2['band'].str.startswith('1')].sort_values('years_of_cover', ascending=False))

band
1 Highest (above 150 years)     6
2 High (120 to 150 years)      53
3 Medium (100 to 120 years)    60
4 Lower (up to 100 years)      61
Name: count, dtype: int64
Total: 180

band    1 Highest (above 150 years)  2 High (120 to 150 years)  \
row_0                                                            
AHM001                            0                          8   
CHN001                            0                          0   
DEL001                            0                         15   
HYD001                            5                         17   
KOL001                            0                          0   
PUN001                            1                         13   

band    3 Medium (100 to 120 years)  4 Lower (up to 100 years)  
row_0                                                           
AHM001                           16                          6  
CHN001                            5                         25  
DEL001                          

In [11]:
used = {
    'so_l':   (so_l,   ['so_id', 'product_id', 'quantity']),
    'so_h':   (so_h,   ['so_id', 'branch_id', 'order_status', 'delivery_date']),
    'po_l':   (po_l,   ['po_id', 'product_id', 'quantity']),
    'po_h':   (po_h,   ['po_id', 'branch_id', 'po_status']),
    'inv':    (inv,    ['product_id', 'branch_id', 'opening_stock', 'current_stock']),
    'ledger': (ledger, ['product_id', 'branch_id', 'movement_type', 'movement_date', 'quantity', 'running_balance']),
}
numeric = {
    'so_l': ['quantity'], 'po_l': ['quantity'],
    'inv': ['opening_stock', 'current_stock'],
    'ledger': ['quantity', 'running_balance'],
}

for name, (df, cols) in used.items():
    line = name + ' | rows: ' + str(len(df)) + ' | empty values: ' + str(int(df[cols].isna().sum().sum()))
    if name in numeric:
        zeros = int((df[numeric[name]] == 0).sum().sum())
        negs = int((df[numeric[name]] < 0).sum().sum())
        line += ' | zeros: ' + str(zeros) + ' | negatives: ' + str(negs)
    print(line)

so_l | rows: 130402 | empty values: 0 | zeros: 0 | negatives: 0
so_h | rows: 20000 | empty values: 0
po_l | rows: 155495 | empty values: 0 | zeros: 0 | negatives: 0
po_h | rows: 24000 | empty values: 0
inv | rows: 180 | empty values: 0 | zeros: 0 | negatives: 0
ledger | rows: 237230 | empty values: 0 | zeros: 0 | negatives: 0


In [12]:
print('ledger | negative quantity:', int((ledger['quantity'] < 0).sum()))
print('ledger | negative running_balance:', int((ledger['running_balance'] < 0).sum()))

ledger | negative quantity: 0
ledger | negative running_balance: 0


In [13]:
all_pairs = pd.MultiIndex.from_frame(inv[['product_id', 'branch_id']])
sold_pairs = pd.MultiIndex.from_frame(sales[['product_id', 'branch_id']].drop_duplicates())

print('Pairs in inventory_master:', len(all_pairs))
print('Pairs missing from the 12-month sales:', len(all_pairs.difference(last_year.index)))
print('Pairs with zero units sold in the window:', int((last_year['units_sold'] == 0).sum()))
print('Sales pairs that are not in inventory_master:', len(sold_pairs.difference(all_pairs)))
print('Pairs with order-line stock of zero or less:', int((stock['order_line_stock'] <= 0).sum()))
print('Smallest units sold in the window:', int(last_year['units_sold'].min()))

Pairs in inventory_master: 180
Pairs missing from the 12-month sales: 0
Pairs with zero units sold in the window: 0
Sales pairs that are not in inventory_master: 0
Pairs with order-line stock of zero or less: 0
Smallest units sold in the window: 703


In [14]:
sales_all = so_l.merge(so_h[['so_id', 'branch_id', 'order_status', 'delivery_date']], on='so_id')
sales_all['delivery_date'] = pd.to_datetime(sales_all['delivery_date'])

print('Order statuses in sales_orders_header:')
print(so_h['order_status'].value_counts())
print()

win = sales_all[sales_all['delivery_date'] > one_year_ago]
print('Order lines and units in the window, by status:')
print(win.groupby('order_status')['quantity'].agg(['count', 'sum']))
print()
print('Delivered units in the window:', int(win.loc[win['order_status'] == 'Delivered', 'quantity'].sum()))
print('Sum of units_sold in last_year:', int(last_year['units_sold'].sum()))
print()

start_day = sales[sales['delivery_date'] == one_year_ago]
print('Delivered order lines on the start day', one_year_ago.date(), ':', len(start_day), '| units:', int(start_day['quantity'].sum()))
print()

for days in [364, 366]:
    start = end_date - pd.Timedelta(days=days)
    u = sales[sales['delivery_date'] > start].groupby(['product_id', 'branch_id'])['quantity'].sum()
    cov2 = (stock['order_line_stock'] / u).reindex(c2.index)
    changed = (cov2.apply(band) != c2['band']).sum()
    print('Window of', days, 'days: pairs that change band:', int(changed))

Order statuses in sales_orders_header:
order_status
Delivered    18033
Cancelled     1967
Name: count, dtype: int64

Order lines and units in the window, by status:
              count     sum
order_status               
Cancelled      2357   24652
Delivered     18889  197300

Delivered units in the window: 197300
Sum of units_sold in last_year: 197300

Delivered order lines on the start day 2024-01-14 : 12 | units: 135

Window of 364 days: pairs that change band: 1
Window of 366 days: pairs that change band: 0


In [16]:
d_ids = set(so_h.loc[so_h['order_status'] == 'Delivered', 'so_id'])
d_date = so_h.set_index('so_id')['delivery_date']

lines = so_l[so_l['so_id'].isin(d_ids)].copy()
lines['d'] = pd.to_datetime(lines['so_id'].map(d_date))

print('Units sold in the window, direct route:', int(lines.loc[lines['d'] > one_year_ago, 'quantity'].sum()))
print('Delivered units, all time:', int(lines['quantity'].sum()), '| KPI 8 figure: 1,234,939')
print()
print('Opening stock, all pairs:', int(stock['opening_stock'].sum()), '| KPI 8 figure: 32,969')
print('Received units, all pairs:', int(stock['received'].sum()), '| KPI 8 figure: 22,387,123')
print('Delivered units, all pairs:', int(stock['delivered'].sum()), '| KPI 8 figure: 1,234,939')
print('Net adjustments, all pairs:', int(stock['net_adj'].sum()), '| KPI 8 figure: 62')
print('Order-line stock, all pairs:', int(stock['order_line_stock'].sum()), '| KPI 8 figure: 21,185,215')
print()
print('Pairs in the cover table:', len(c2), '| pairs in the bands:', int(c2['band'].value_counts().sum()))

Units sold in the window, direct route: 197300
Delivered units, all time: 1234939 | KPI 8 figure: 1,234,939

Opening stock, all pairs: 32969 | KPI 8 figure: 32,969
Received units, all pairs: 22387123 | KPI 8 figure: 22,387,123
Delivered units, all pairs: 1234939 | KPI 8 figure: 1,234,939
Net adjustments, all pairs: 62 | KPI 8 figure: 62
Order-line stock, all pairs: 21185215 | KPI 8 figure: 21,185,215

Pairs in the cover table: 180 | pairs in the bands: 180


In [17]:
# Route B: pivot table straight from the raw delivered order lines in the window
w = lines[lines['d'] > one_year_ago].copy()
w['branch_id'] = w['so_id'].map(so_h.set_index('so_id')['branch_id'])
pv = w.pivot_table(index='product_id', columns='branch_id', values='quantity', aggfunc='sum')

cover_b = stock['order_line_stock'].unstack('branch_id') / pv
cover_a = c2['years_of_cover'].unstack('branch_id')

print('Cells compared:', cover_b.size)
print('Largest difference between the two routes (years):', float((cover_b - cover_a).abs().max().max()))
print()

# Ledger-based stock for comparison
ledger_last = led.groupby(['product_id', 'branch_id'])['running_balance'].last()
print('Ledger-based stock, all pairs:', int(ledger_last.sum()), '| KPI 8 figure: 19,305,994')

cover_ledger = ledger_last / c2['units_sold']
print('Ledger-based years of cover: mean', round(cover_ledger.mean(), 1), '| min', round(cover_ledger.min(), 1), '| max', round(cover_ledger.max(), 1))
print()
print('Bands with ledger-based stock:')
print(cover_ledger.apply(band).value_counts().sort_index())

Cells compared: 180
Largest difference between the two routes (years): 0.0

Ledger-based stock, all pairs: 19305994 | KPI 8 figure: 19,305,994
Ledger-based years of cover: mean 100.7 | min 64.8 | max 157.9

Bands with ledger-based stock:
1 Highest (above 150 years)     3
2 High (120 to 150 years)      24
3 Medium (100 to 120 years)    57
4 Lower (up to 100 years)      96
Name: count, dtype: int64


In [18]:
y = c2['years_of_cover']
q1, q3 = y.quantile(0.25), y.quantile(0.75)
iqr = q3 - q1
low_fence, high_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
print('Q1:', round(q1, 1), '| Q3:', round(q3, 1), '| normal range:', round(low_fence, 1), 'to', round(high_fence, 1))
print('Pairs below the range:', int((y < low_fence).sum()), '| pairs above the range:', int((y > high_fence).sum()))
print()

for edge in [100, 120, 150]:
    print('Pairs within 2 years of', edge, ':', int(((y - edge).abs() <= 2).sum()))
print()

top_order = set(y.sort_values(ascending=False).head(6).index)
top_ledger = set(cover_ledger.sort_values(ascending=False).head(6).index)
print('Worst 6 by order-line stock that are also in the worst 6 by ledger stock:', len(top_order & top_ledger), 'of 6')
print('Worst 6 by ledger stock:')
print(cover_ledger.sort_values(ascending=False).head(6).round(1))

Q1: 96.3 | Q3: 124.4 | normal range: 54.2 to 166.5
Pairs below the range: 0 | pairs above the range: 1

Pairs within 2 years of 100 : 12
Pairs within 2 years of 120 : 9
Pairs within 2 years of 150 : 2

Worst 6 by order-line stock that are also in the worst 6 by ledger stock: 6 of 6
Worst 6 by ledger stock:
product_id  branch_id
P005        HYD001       157.9
P015        PUN001       150.9
P004        HYD001       150.8
P018        HYD001       139.6
P002        HYD001       137.5
P019        HYD001       134.6
dtype: float64


In [19]:
print('First delivery date in the window:', w['d'].min().date())
print('Last delivery date in the window:', w['d'].max().date())
print()

w['month'] = w['d'].dt.to_period('M')
print('Delivered order lines per month and branch:')
print(w.pivot_table(index='month', columns='branch_id', values='quantity', aggfunc='count'))
print()

print('Last delivery date per branch (all history):')
print(sales.groupby('branch_id')['delivery_date'].max().dt.date)

First delivery date in the window: 2024-01-15
Last delivery date in the window: 2025-01-13

Delivered order lines per month and branch:
branch_id  AHM001  CHN001  DEL001  HYD001  KOL001  PUN001
month                                                    
2024-01       136     165     206     136     215     110
2024-02       319     324     306     173     235     216
2024-03       209     432     246     260     274     244
2024-04       242     240     187     259     335     222
2024-05       211     366     251     184     254     278
2024-06       250     328     196     182     278     230
2024-07       332     360     211     227     293     266
2024-08       308     331     292     176     305     247
2024-09       224     263     241     296     298     273
2024-10       281     278     288     284     298     361
2024-11       258     386     197     188     286     243
2024-12       225     317     299     183     371     164
2025-01        52      41      45      58      69   

In [20]:
so_date = so_h.set_index('so_id')['delivery_date']
median_cover = c2['years_of_cover'].median()
mid = (c2['years_of_cover'] - median_cover).abs().idxmin()

for p, b in [('P005', 'HYD001'), ('P027', 'CHN001'), mid]:
    del_ids = set(so_h.loc[(so_h['branch_id'] == b) & (so_h['order_status'] == 'Delivered'), 'so_id'])
    sl = so_l[(so_l['product_id'] == p) & (so_l['so_id'].isin(del_ids))].copy()
    sl['d'] = pd.to_datetime(sl['so_id'].map(so_date))
    sold_all = int(sl['quantity'].sum())
    sold_win = int(sl.loc[sl['d'] > one_year_ago, 'quantity'].sum())

    rec_ids = set(po_h.loc[(po_h['branch_id'] == b) & (po_h['po_status'] == 'Received'), 'po_id'])
    received = int(po_l[(po_l['product_id'] == p) & (po_l['po_id'].isin(rec_ids))]['quantity'].sum())

    opening = int(inv[(inv['product_id'] == p) & (inv['branch_id'] == b)]['opening_stock'].iloc[0])
    adj = led[(led['product_id'] == p) & (led['branch_id'] == b) & (led['movement_type'] == 'ADJUSTMENT')]
    net = int(adj['change'].sum())

    hand_stock = opening + received - sold_all + net
    hand_years = hand_stock / sold_win

    print('Pair', p, b)
    print('  By hand: opening', opening, '| received', received, '| delivered all time', sold_all, '| adjustment rows', len(adj), '| net adjustments', net)
    print('  By hand: units sold in window', sold_win, '| stock', hand_stock, '| years of cover', round(hand_years, 2))
    print('  Table:   units sold in window', int(c2.loc[(p, b), 'units_sold']), '| stock', int(c2.loc[(p, b), 'order_line_stock']), '| years of cover', round(c2.loc[(p, b), 'years_of_cover'], 2))
    print()

Pair P005 HYD001
  By hand: opening 178 | received 128466 | delivered all time 5663 | adjustment rows 19 | net adjustments -57
  By hand: units sold in window 715 | stock 122924 | years of cover 171.92
  Table:   units sold in window 715 | stock 122924 | years of cover 171.92

Pair P027 CHN001
  By hand: opening 84 | received 117598 | delivered all time 8628 | adjustment rows 8 | net adjustments -65
  By hand: units sold in window 1513 | stock 108989 | years of cover 72.04
  Table:   units sold in window 1513 | stock 108989 | years of cover 72.04

Pair P015 AHM001
  By hand: opening 85 | received 123103 | delivered all time 7250 | adjustment rows 16 | net adjustments -28
  By hand: units sold in window 1074 | stock 115910 | years of cover 107.92
  Table:   units sold in window 1074 | stock 115910 | years of cover 107.92

